# 🎬 Bilibili 影片下載 + Markdown 轉換器 v3.0

使用 **bilix** 專業下載工具，先下載影片到 Google Drive，再轉錄成 Markdown。

**作者**：蝦仔  
**版本**：v3.0 (使用 bilix + Whisper)

## 📋 流程說明

```
Step 1: 連接 Google Drive
    ↓
Step 2: 安裝 bilix (專業 Bilibili 下載器)
    ↓
Step 3: 下載影片到 Google Drive
    ↓
Step 4: 安裝 Whisper
    ↓
Step 5: 語音轉錄
    ↓
Step 6: 生成 Markdown (儲存在 Google Drive)
```

**優點**：
- ✅ bilix 專為 Bilibili 設計，下載更穩定
- ✅ 支援 1080P 高畫質
- ✅ 影片先儲存在 Google Drive，可重複使用
- ✅ 分離下載同轉錄流程，更靈活

---

## Step 1: 連接 Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# 設定工作目錄
WORK_DIR = "/content/drive/MyDrive/bilibili-projects"
VIDEO_DIR = f"{WORK_DIR}/videos"
AUDIO_DIR = f"{WORK_DIR}/audio"
OUTPUT_DIR = f"{WORK_DIR}/markdown"

os.makedirs(VIDEO_DIR, exist_ok=True)
os.makedirs(AUDIO_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

%cd {WORK_DIR}
print("✅ Google Drive 連接成功！")
print(f"📁 影片目錄: {VIDEO_DIR}")
print(f"📁 Markdown目錄: {OUTPUT_DIR}")

---

## Step 2: 安裝下載工具 (bilix)

In [ ]:
print("📦 安裝 bilix (Bilibili 專業下載器)...")
!pip install -q bilix

print("\n✅ bilix 安裝完成！")
print("\n📖 使用方法:")
print("  bilix v 'https://www.bilibili.com/video/BVxxxxx'")
print("  bilix v 'https://space.bilibili.com/xxx/lists/xxx'  # 播放清單")

---

## Step 3: 配置並下載影片

In [ ]:
# ==========================================
# 請修改以下設定
# ==========================================

# Bilibili 播放清單或影片 URL
VIDEO_URL = "https://space.bilibili.com/1925268550/lists/839662?type=season"

# 畫質選擇: 80 (1080P), 64 (720P), 32 (480P), 16 (360P)
QUALITY = 80

# 下載數量限制 (None = 全部, 數字 = 只下載前 N 個)
DOWNLOAD_LIMIT = 5

# ==========================================

print("📝 下載設定:")
print(f"  URL: {VIDEO_URL}")
print(f"  畫質: {QUALITY}P")
print(f"  限制: {DOWNLOAD_LIMIT if DOWNLOAD_LIMIT else '無限制'}")
print(f"  儲存位置: {VIDEO_DIR}")

In [ ]:
import subprocess
import re

def download_with_bilix(url, output_dir, quality=80, limit=None):
    """
    使用 bilix 下載 Bilibili 影片
    """
    cmd = [
        "bilix",
        "v",  # get_video 嘅簡寫
        url,
        "--dir", output_dir,
        "--quality", str(quality)
    ]
    
    if limit:
        cmd.extend(["--num", str(limit)])
    
    print(f"\n⬇️  開始下載...")
    print(f"   指令: {' '.join(cmd)}\n")
    print("="*60)
    
    # 運行下載
    result = subprocess.run(cmd, capture_output=True, text=True)
    
    print("="*60)
    
    if result.returncode != 0:
        print(f"❌ 下載出錯:")
        print(result.stderr)
        return []
    
    # 解析下載結果
    print(result.stdout)
    
    return result.stdout

# 執行下載
download_output = download_with_bilix(
    VIDEO_URL, 
    VIDEO_DIR, 
    quality=QUALITY,
    limit=DOWNLOAD_LIMIT
)

# 檢查下載嘅檔案
import glob
downloaded_videos = glob.glob(f"{VIDEO_DIR}/**/*.mp4", recursive=True) + \\n                      glob.glob(f"{VIDEO_DIR}/**/*.flv", recursive=True) + \\n                      glob.glob(f"{VIDEO_DIR}/**/*.mkv", recursive=True)

print(f"\n📊 下載結果:")
print(f"  找到 {len(downloaded_videos)} 個影片檔案")

for i, v in enumerate(downloaded_videos, 1):
    size_mb = os.path.getsize(v) / (1024*1024)
    print(f"  {i}. {os.path.basename(v)} ({size_mb:.1f} MB)")

---

## Step 4: 安裝 Whisper (語音轉文字)

In [ ]:
print("📦 安裝 Whisper...")
!pip install -q openai-whisper
!apt-get update -qq && apt-get install -y -qq ffmpeg

print("\n✅ Whisper 安裝完成！")

---

## Step 5: 配置轉錄設定

In [ ]:
# ==========================================
# Whisper 設定
# ==========================================

# Whisper 模型: tiny, base, small, medium, large
WHISPER_MODEL = "small"

# 語言: zh (中文), en (英文), ja (日文), ko (韓文)
LANGUAGE = "zh"

# ==========================================

print("🎯 轉錄設定:")
print(f"  模型: {WHISPER_MODEL}")
print(f"  語言: {LANGUAGE}")
print(f"  來源: {VIDEO_DIR}")
print(f"  輸出: {OUTPUT_DIR}")

---

## Step 6: 語音轉錄並生成 Markdown

In [ ]:
import whisper
import subprocess
from pathlib import Path
from datetime import datetime

# 載入 Whisper 模型
print(f"🤖 載入 Whisper 模型: {WHISPER_MODEL}...")
model = whisper.load_model(WHISPER_MODEL)
print("✅ 模型載入完成！\n")

def extract_audio(video_path, audio_path):
    """提取音軌"""
    cmd = [
        "ffmpeg", "-i", video_path,
        "-vn", "-acodec", "pcm_s16le",
        "-ar", "16000", "-ac", "1",
        "-y", audio_path
    ]
    subprocess.run(cmd, capture_output=True, check=True)
    return audio_path

def transcribe_video(video_path, model):
    """轉錄影片"""
    # 生成音訊檔案路徑
    video_name = Path(video_path).stem
    audio_path = f"{AUDIO_DIR}/{video_name}.wav"
    
    # 提取音軌
    print(f"  🔊 提取音軌...")
    extract_audio(video_path, audio_path)
    
    # 轉錄
    print(f"  🎯 轉錄中...")
    result = model.transcribe(audio_path, language=LANGUAGE)
    
    return result

def generate_markdown(video_path, transcript, output_dir):
    """生成 Markdown 檔案"""
    video_name = Path(video_path).stem
    md_path = f"{output_dir}/{video_name}.md"
    
    # 獲取影片資訊
    size_mb = os.path.getsize(video_path) / (1024*1024)
    
    content = f"""# {video_name}

## 元數據

| 項目 | 內容 |
|------|------|
| **檔案名稱** | {video_name} |
| **檔案大小** | {size_mb:.1f} MB |
| **來源** | {video_path} |
| **生成日期** | {datetime.now().strftime('%Y-%m-%d %H:%M')} |

---

## 內容轉錄

"""
    
    # 添加時間戳段落
    for seg in transcript.get('segments', []):
        start_m, start_s = int(seg['start'] // 60), int(seg['start'] % 60)
        end_m, end_s = int(seg['end'] // 60), int(seg['end'] % 60)
        text = seg['text'].strip()
        
        if text:
            content += f"### {start_m:02d}:{start_s:02d} - {end_m:02d}:{end_s:02d}\n\n"
            content += f"{text}\n\n"
    
    content += "---\n\n*由 bilibili-to-markdown 自動生成*\n"
    
    with open(md_path, 'w', encoding='utf-8') as f:
        f.write(content)
    
    return md_path

# 處理所有影片
print("\n" + "="*60)
print("📝 開始轉錄影片")
print("="*60 + "\n")

results = []

for i, video_path in enumerate(downloaded_videos, 1):
    video_name = Path(video_path).name
    print(f"\n[{i}/{len(downloaded_videos)}] {video_name}")
    
    try:
        # 轉錄
        transcript = transcribe_video(video_path, model)
        
        # 生成 Markdown
        md_path = generate_markdown(video_path, transcript, OUTPUT_DIR)
        
        results.append({
            'video': video_path,
            'markdown': md_path,
            'title': Path(video_path).stem
        })
        
        print(f"  ✅ 完成: {Path(md_path).name}")
        
    except Exception as e:
        print(f"  ❌ 失敗: {e}")

print(f"\n📊 轉錄完成: {len(results)}/{len(downloaded_videos)}")

---

## Step 7: 生成索引檔案

In [ ]:
# 生成 summary.md 索引
summary_path = f"{OUTPUT_DIR}/summary.md"

with open(summary_path, 'w', encoding='utf-8') as f:
    f.write("# Bilibili 影片轉錄索引\n\n")
    f.write(f"**生成時間**: {datetime.now().strftime('%Y-%m-%d %H:%M')}\n\n")
    f.write(f"**總影片數**: {len(results)}\n\n")
    f.write("## 影片列表\n\n")
    f.write("| 序號 | 標題 |\n")
    f.write("|------|------|\n")
    
    for i, r in enumerate(results, 1):
        md_filename = Path(r['markdown']).name
        f.write(f"| {i} | [{r['title']}](./{md_filename}) |\n")
    
    f.write("\n---\n\n")
    f.write("*由 bilibili-to-markdown 自動生成*\n")

print(f"✅ 索引檔案: {summary_path}")

---

## ✅ 全部完成！

In [ ]:
print("\n" + "="*60)
print("🎉 處理完成！")
print("="*60)

print(f"\n📁 Google Drive 檔案位置:")
print(f"  📹 影片: {VIDEO_DIR}")
print(f"  📝 Markdown: {OUTPUT_DIR}")

print(f"\n📊 統計:")
print(f"  - 下載影片: {len(downloaded_videos)}")
print(f"  - 轉錄完成: {len(results)}")

print(f"\n📄 檔案列表:")
for r in results:
    print(f"  - {Path(r['markdown']).name}")

print(f"\n💡 提示: 去 Google Drive 查看結果")
print("="*60)

---

## 🔄 進階：單獨運行轉錄 (可選)

如果你已經有影片在 Google Drive，可以直接運行呢段代碼做轉錄：

In [ ]:
# 只運行轉錄（不重新下載）
# 確保 VIDEO_DIR 入面有影片

# import glob
# existing_videos = glob.glob(f"{VIDEO_DIR}/**/*.mp4", recursive=True)
# print(f"找到 {len(existing_videos)} 個現有影片")
# 
# 然後運行 Step 6 嘅轉錄代碼...